## Ingest Dimention Data into Silver

In [0]:
# Import required libraries
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, FloatType

In [0]:
%run /Workspace/Users/yklynk@gmail.com/Azure_databricks_data_engineering_project_shopvista_ecomm/1_setup/utilities

In [0]:
print (bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text('catalog','shopvista', 'catalog')
dbutils.widgets.text('brands_source', 'brands', 'brands_source' )
dbutils.widgets.text('category_source', 'category', 'category_source')
dbutils.widgets.text('customers_source', 'customers', 'customers_source' )
dbutils.widgets.text('date_source', 'date', 'date_source')
dbutils.widgets.text('product_source', 'products', 'product_source')

## BRANDS

In [0]:
# Retrieve and Print Catalog & Brands Source Widget Values
catalog = dbutils.widgets.get('catalog')
brands_source = dbutils.widgets.get('brands_source')

print(catalog, brands_source)

In [0]:
df_bronze = spark.table(f"{catalog}.{bronze_schema}.{brands_source}")
display(df_bronze.limit(5))

- Data Cleaning and Transformation

In [0]:
# Trim brand name column to remove spaces
df_silver = df_bronze.withColumn("brand_name", F.trim(F.col("brand_name")))
display(df_silver.limit(5))

In [0]:
# Remove all non-alphanumeric characters from brand_code
df_silver = df_silver.withColumn("brand_code",F.regexp_replace(F.col("brand_code"), r"[^a-zA-Z0-9]", ""))
display(df_silver.limit(10))

In [0]:
# Show the distinct values in category_code

display(df_silver.select("category_code").distinct())

In [0]:
# Replace the incorrect category codes with the right one
# Anomalies dictionary
anomalies = {
    "GROCERY": "GRCY",
    "BOOKS": "BKS",
    "TOYS": "TOY"
}

# PySpark replace is easy
df_silver = df_silver.replace(to_replace=anomalies, subset=["category_code"])

# ✅ Show results
df_silver.select("category_code").distinct().show()

In [0]:
# write to silver layer
df_silver.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed', True)\
    .option('mergeschema', 'true')\
    .mode('overwrite')\
    .saveAsTable(f'{catalog}.{silver_schema}.{brands_source}')


## CATEGORY

In [0]:
# Retrieve and Print Catalog & Category Source Widget Values
catalog = dbutils.widgets.get('catalog')
category_source = dbutils.widgets.get('category_source')

print(catalog, category_source)

In [0]:
# Read Table
df_bronze = spark.table(f"{catalog}.{bronze_schema}.{category_source}")
display(df_bronze.limit(5))

- Category Cleaning and Transformation

In [0]:
# check for duplicates
duplicates_df = df_bronze.groupBy("category_code").count()
display(duplicates_df)

In [0]:
# Detect duplicates
duplicates_df = df_bronze.groupBy("category_code").count().filter(F.col("count") > 1)
display(duplicates_df)

In [0]:
#Remove duplicates
df_silver = df_bronze.dropDuplicates(["category_code"])
display(df_silver)

In [0]:
# Change category code format from lower case to  upper case align with 
df_silver = df_silver.withColumn("category_code", F.upper(F.col("category_code")))

display(df_silver.limit(5))

In [0]:
# write to silver table
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", True)\
    .mode('overwrite')\
    .saveAsTable(f"{catalog}.{silver_schema}.{category_source}")

## CUSTOMERS

In [0]:
# Retrieve and Print Catalog & Customers Source Widget Values
catalog = dbutils.widgets.get('catalog')
customers_source = dbutils.widgets.get('customers_source')

print(catalog, customers_source)

In [0]:
# load table
df_bronze = spark.table(f"{catalog}.{bronze_schema}.{customers_source}")

display(df_bronze.limit(5))

- Customer Cleaning and Transformation

In [0]:
# check null values in customer_id, phone, country_code,country, state
null_customer  = df_bronze.filter(F.col("customer_id").isNull()).count()
print(null_customer)
null_phone = df_bronze.filter(F.col("phone").isNull()).count()
print(null_phone)
null_countrycode = df_bronze.filter(F.col("country_code").isNull()).count()
print(null_countrycode)
null_country = df_bronze.filter(F.col("country").isNull()).count()
print(null_country)
null_state = df_bronze.filter(F.col("state").isNull()).count()
print(null_state)

In [0]:
# There are 300 null values in customer_id column, show some
df_bronze.filter(F.col("customer_id").isNull()).show(5)

In [0]:
# Drop rows where 'customer_id' is null
df_silver = df_bronze.dropna(subset=["customer_id"])

# Get row count
row_count = df_silver.count()
print(f"Row count after droping null values: {row_count}")

In [0]:
# There are over 3000 null values in phone column, show some
df_silver.filter(F.col("phone").isNull()).show(5)

In [0]:
# Handle null values in phone column (we have 3000 null values in phone columm)
df_silver = df_silver.fillna('Not Availabe', subset=["phone"])

# check to confirm
df_silver.filter(F.col("phone").isNull()).show()

In [0]:
# write to silver table
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDatafeed", True)\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{customers_source}")

In [0]:
display(df_silver.limit(5))

## DATE

In [0]:
# Retrieve and Print Catalog & Date Source Widget Values
catalog = dbutils.widgets.get('catalog')
date_source = dbutils.widgets.get('date_source')
print(catalog, date_source)

In [0]:
df_bronze = spark.table(f'{catalog}.{bronze_schema}.{date_source}')
display(df_bronze.limit(5))

- Date Cleaning and Transformation

In [0]:
# check for Null values in the columns
null_date = df_bronze.filter(F.col('date').isNull()).count()
print(null_date)

null_year = df_bronze.filter(F.col('year').isNull()).count()
print(null_year)

null_quarter = df_bronze.filter(F.col('quarter').isNull()).count()
print(null_quarter)

null_week = df_bronze.filter(F.col('week_of_year').isNull()).count()
print(null_week)

In [0]:
# Remove duplicates in the columns
duplicates = df_bronze.groupby('date').count().filter('count > 1')
# Show the duplicate rows
print("Total duplicated Rows: ", duplicates.count())
display(duplicates)

In [0]:
# Remove rows with duplicates
df_silver = df_bronze.dropDuplicates(['date'])
df_silver.count()

In [0]:
# convert first letter of day_name column to upper case
df_silver = df_silver.withColumn('day_name', F.initcap(F.col('day_name')))
display(df_silver)

In [0]:
# Convert negative values in 'week_of_year' column to positive
df_silver = df_silver.withColumn('week_of_year', F.abs(F.col('week_of_year')))
display(df_silver.limit(5))

In [0]:
# Enhance the quarter and week of year colum
df_silver = df_silver.withColumn("quarter", F.concat_ws("", F.concat(F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year"))))

df_silver = df_silver.withColumn("week_of_year", F.concat_ws("-", F.concat(F.lit("Week"), F.col("week_of_year"), F.lit("-"), F.col("year"))))

display(df_silver.limit(5))

In [0]:
# Rename week of year column to week
df_silver = df_silver.withColumnRenamed("week_of_year", "week")

display(df_silver.limit(5))

In [0]:
# write to silver table
df_silver.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed', True)\
    .mode('overwrite')\
    .saveAsTable(f'{catalog}.{silver_schema}.{date_source}')

In [0]:
display(df_silver.limit(5))

## PRODUCT

In [0]:
# Retrieve and Print Catalog & Products Source Widget Values
catalog = dbutils.widgets.get('catalog')
product_source = dbutils.widgets.get('product_source')

print(catalog, product_source)

In [0]:
df_bronze = spark.table(f'{catalog}.{bronze_schema}.{product_source}')

- Product Cleaning and Transformation

In [0]:
# check null value in the columns
null_producd = df_bronze.filter(F.col('product_id').isNull()).count()
print(null_producd)
null_skU = df_bronze.filter(F.col('sku').isNull()).count()
print(null_skU)
null_category = df_bronze.filter(F.col('category_code').isNull()).count()
print(null_category) 
null_brand =  df_bronze.filter(F.col('brand_code').isNull()).count()     
print(null_brand)
null_color =  df_bronze.filter(F.col('color').isNull()).count()   
print(null_color)     
null_size =  df_bronze.filter(F.col('size').isNull()).count()  
print(null_size)           
null_material =  df_bronze.filter(F.col('material').isNull()).count()
print(null_material)  
null_weight =  df_bronze.filter(F.col('weight_grams').isNull()).count()
print(null_weight)  
null_lenght =  df_bronze.filter(F.col('length_cm').isNull()).count()  
print(null_lenght)
null_width =  df_bronze.filter(F.col('width_cm').isNull()).count()
print(null_width)
null_height =  df_bronze.filter(F.col('width_cm').isNull()).count()
print(null_height)
null_rating =  df_bronze.filter(F.col('rating_count').isNull()).count()
print(null_rating)

In [0]:
# show rows that has null values in the color colums
null_color =  df_bronze.filter(F.col('color').isNull())
display(null_color.limit(5))

In [0]:
# fill null values in colour with 'Not Available'

df_silver = df_bronze.fillna('Not Available', subset=['color'])
display(df_silver.limit(5))

In [0]:
# sanity check
df_silver.filter(F.col('color').isNull()).count()

In [0]:
# check weight_gram contain 'g'
df_silver.select("weight_grams").show(5, truncate=False)


In [0]:
# Remove 'g'from the weigt_gram colum

df_silver = df_silver.withColumn('weight_grams', F.regexp_replace('weight_grams', 'g', '').cast(FloatType()))
display(df_silver.select('weight_grams').limit(5))

In [0]:
# change comma in leght_cm values to dot
df_silver = df_silver.withColumn('length_cm', F.regexp_replace('length_cm', ',', '.').cast(FloatType()))
display(df_silver.limit(5))

In [0]:
# change category code and brand code to upper case
df_silver = df_silver.withColumn('category_code', F.upper(F.col('category_code')))
df_silver = df_silver.withColumn('brand_code', F.upper(F.col('brand_code')))

In [0]:
display(df_silver.limit(5))

In [0]:
df_silver.select("material").distinct().show()

In [0]:
# Fix spelling mistakes
df_silver = df_silver.withColumn(
    "material",
    F.when(F.col("material") == "Coton", "Cotton")
     .when(F.col("material") == "Alumium", "Aluminium")
     .when(F.col("material") == "Ruber", "Rubber")
     .otherwise(F.col("material"))
)

df_silver.select('material').distinct().show()

In [0]:
# Negative values in rating count
df_silver.filter(F.col('rating_count')<0).select('rating_count').show(3)

In [0]:
df_silver = df_silver.withColumn(
    "rating_count",
    F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count")))
     .otherwise(F.lit(0))  # if null, replace with 0
)

display(df_silver.limit(5))

In [0]:
#check all cleaned data

df_silver.select(
    "weight_grams",
    "length_cm",
    "category_code",
    "brand_code",
    "material",
    "rating_count"
).show(10, truncate=False)

In [0]:
# write to the silver table
df_silver.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed', True)\
    .mode('overwrite')\
    .saveAsTable(f'{catalog}.{silver_schema}.{product_source}')   


In [0]:
display(df_silver.limit(5))